# Data Engineering Mini-Project 1
## Student Success & Early-Warning Analytics System — Raw-to-Production Data Pipeline

**Name:** Horn Seyhak

**Course:** CLD 375



In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 140)
audit = {}


## 1. Data Ingestion

We load the raw CSV file and the data dictionary (which defines each column's expected type and valid-value rule). The data dictionary is our single source of truth for every decision made later in the pipeline.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
raw_path = "/content/drive/MyDrive/Could_Ml_classwork/project 1/university_student_success_raw_2500x25.csv"
dict_path = "/content/drive/MyDrive/Could_Ml_classwork/project 1/university_student_success_data_dictionary.xlsx"

df = pd.read_csv(raw_path)
data_dict = pd.read_excel(dict_path)

print("Raw dataset shape:", df.shape)
df.head()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Raw dataset shape: (2500, 25)


,Student_ID,Admission_Year,Age,Gender,Program,Study_Year,Semester,Attendance_Percent,Study_Hours_Per_Week,LMS_Logins_Per_Week,Assignment_Completion_Percent,Average_Assignment_Score,Midterm_Score,Final_Exam_Score,Current_GPA,Credits_Registered,Credits_Earned,Failed_Courses,Backlog_Courses,Financial_Aid_Status,Scholarship_Amount,Library_Visits_Per_Month,Counseling_Visits,Extracurricular_Level,Academic_Risk_Level
0,20260617.0,2023.0,19,Male,NaN,2.0,4.0,87.2,7.6,NaN,61.0,76.6,61.1,65.5,NaN,16,16.0,0.0,0.0,Full,2500.0,4.0,4.0,Low,Low
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20261481.0,2025.0,24,Other,Electrical Engineering,3.0,6.0,74.8,13.3,10.0,100.0,84.7,47.7,76.0,3.72,16,13.0,1.0,1.0,NaN,NaN,3.0,2.0,Low,Medium
3,20260538.0,2023.0,24,Male,Civil Engineering,4.0,8.0,79.3,12.0,6.0,98.2,84.7,76.9,53.0,4.0,24,21.0,0.0,0.0,NaN,NaN,NaN,0.0,Low,Low
4,20261127.0,2024.0,22,NaN,Electrical Engineering,3.0,5.0,89.5,22.2,8.0,100.0,80.3,63.8,81.4,3.96,16,16.0,0.0,1.0,Partial,NaN,4.0,0.0,Low,Low


In [3]:
data_dict


,Column,Type,Business_Meaning,Expected_Rule
0,Student_ID,Identifier,Unique ID; never impute,Integer
1,Admission_Year,Temporal,Admission year,2022–2026
2,Age,Numeric,Age in years,Normally 18–30
3,Gender,Nominal,Gender category,Male/Female/Other
4,Program,Nominal,Degree program,Categorical
5,Study_Year,Ordinal/Discrete,Year of study,1–4
6,Semester,Ordinal/Discrete,Current semester,1–8
7,Attendance_Percent,Continuous,Class attendance,0–100
8,Study_Hours_Per_Week,Continuous,Weekly independent study,>=0
9,LMS_Logins_Per_Week,Discrete,Weekly LMS logins,>=0


## 2. Data Profiling

### 2.1 Structure and types


In [4]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2500 entries, 0 to 2499
Data columns (total 25 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Student_ID                     2470 non-null   float64
 1   Admission_Year                 2470 non-null   float64
 2   Age                            2470 non-null   object 
 3   Gender                         2324 non-null   object 
 4   Program                        2374 non-null   object 
 5   Study_Year                     2470 non-null   float64
 6   Semester                       2470 non-null   float64
 7   Attendance_Percent             2470 non-null   object 
 8   Study_Hours_Per_Week           2298 non-null   object 
 9   LMS_Logins_Per_Week            2324 non-null   float64
 10  Assignment_Completion_Percent  2272 non-null   float64
 11  Average_Assignment_Score       2470 non-null   float64
 12  Midterm_Score                  2470 non-null   f

### 2.2 Descriptive statistics

In [5]:
df.describe(include='all').T


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Student_ID,2470.0,NaN,NaN,NaN,20261206.131984,695.709222,20260001.0,20260603.25,20261205.5,20261805.75,20262410.0
Admission_Year,2470.0,NaN,NaN,NaN,2023.946964,1.411641,2022.0,2023.0,2024.0,2025.0,2026.0
Age,2470,14,21,549,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gender,2324,5,Male,809,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Program,2374,8,Civil Engineering,336,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Study_Year,2470.0,NaN,NaN,NaN,2.349393,1.096678,0.0,1.0,2.0,3.0,5.0
Semester,2470.0,NaN,NaN,NaN,4.2,2.251969,0.0,2.0,4.0,6.0,10.0
Attendance_Percent,2470,408,100.0,116,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Study_Hours_Per_Week,2298,303,2.0,65,NaN,NaN,NaN,NaN,NaN,NaN,NaN
LMS_Logins_Per_Week,2324.0,NaN,NaN,NaN,12.167814,4.536007,-3.0,10.0,12.0,14.0,150.0


### 2.3 Variable classification

Using the `Type` column from the data dictionary, every column is classified below. This matters because each type gets a **different** missing-value and outlier-treatment strategy later in the pipeline.

| Column | Type |
|---|---|
| Student_ID | Identifier (never imputed / never used as a feature) |
| Admission_Year | Temporal / Discrete |
| Age | Continuous (Numeric) |
| Gender | Nominal |
| Program | Nominal |
| Study_Year | Ordinal / Discrete |
| Semester | Ordinal / Discrete |
| Attendance_Percent | Continuous |
| Study_Hours_Per_Week | Continuous |
| LMS_Logins_Per_Week | Discrete |
| Assignment_Completion_Percent | Continuous |
| Average_Assignment_Score | Continuous |
| Midterm_Score | Continuous |
| Final_Exam_Score | Continuous |
| Current_GPA | Continuous |
| Credits_Registered | Discrete |
| Credits_Earned | Discrete |
| Failed_Courses | Discrete |
| Backlog_Courses | Discrete |
| Financial_Aid_Status | Nominal |
| Scholarship_Amount | Continuous |
| Library_Visits_Per_Month | Discrete |
| Counseling_Visits | Discrete |
| Extracurricular_Level | Ordinal (None < Low < Medium < High) |
| Academic_Risk_Level | Ordinal (Low < Medium < High) |


### 2.4 Missing values, duplicates, and blank rows — first look

In [6]:
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
print(missing_pct)

print("\nCompletely blank rows:", df.isnull().all(axis=1).sum())
print("Exact duplicate rows:", df.duplicated().sum())


Scholarship_Amount               75.32
Financial_Aid_Status             60.24
Extracurricular_Level            40.64
Library_Visits_Per_Month         10.08
Assignment_Completion_Percent     9.12
Failed_Courses                    8.28
Study_Hours_Per_Week              8.08
Gender                            7.04
LMS_Logins_Per_Week               7.04
Current_GPA                       6.12
Academic_Risk_Level               5.12
Program                           5.04
Semester                          1.20
Age                               1.20
Student_ID                        1.20
Study_Year                        1.20
Admission_Year                    1.20
Attendance_Percent                1.20
Credits_Earned                    1.20
Final_Exam_Score                  1.20
Midterm_Score                     1.20
Average_Assignment_Score          1.20
Credits_Registered                1.20
Backlog_Courses                   1.20
Counseling_Visits                 1.20
dtype: float64

Completel

### 2.5 Uncovering hidden data-quality problems

Printing the unique values of every text column reveals problems that `.info()` and `.describe()` hide: numbers typed as words, extra whitespace, inconsistent casing, and impossible values.


In [7]:
obj_cols = df.select_dtypes(include='object').columns.tolist()
for c in obj_cols:
    print(f"--- {c} ({df[c].nunique()} unique) ---")
    print(df[c].value_counts(dropna=False).head(10))
    print()


--- Age (14 unique) ---
Age
21     549
20     502
22     450
19     325
23     268
18     145
24     138
17      53
NaN     30
25      25
Name: count, dtype: int64

--- Gender (5 unique) ---
Gender
Male      809
Female    759
Other     754
NaN       176
FEMALE      1
 male       1
Name: count, dtype: int64

--- Program (8 unique) ---
Program
Civil Engineering          336
Electronics Engineering    312
Data Science               311
Business Administration    301
Electrical Engineering     294
Information Technology     293
Computer Science           264
Mechanical Engineering     263
NaN                        126
Name: count, dtype: int64

--- Attendance_Percent (408 unique) ---
Attendance_Percent
100.0    116
NaN       30
84.6      17
77.1      17
87.4      17
84.0      16
82.9      16
85.8      16
76.5      16
83.1      16
Name: count, dtype: int64

--- Study_Hours_Per_Week (303 unique) ---
Study_Hours_Per_Week
NaN     202
2.0      65
11.8     20
12.4     20
13.8     18
13.0     18

### 2.6 Summary of data-quality issues found

What I found:

| Issue | Evidence |
|---|---|
| **Duplicates** | 89 exact duplicate rows |
| **Completely blank rows** | 30 rows with every column empty |
| **High-missing column** | `Scholarship_Amount` is 75.3% missing (> 70%) |
| **Nominal missing values** | `Gender` (7.0%), `Program` (5.0%), `Financial_Aid_Status` (60.2%) |
| **Ordinal missing values** | `Academic_Risk_Level` (5.1%), `Extracurricular_Level` (40.6%) |
| **Discrete missing values** | `Failed_Courses`, `LMS_Logins_Per_Week`, `Library_Visits_Per_Month`, `Study_Year`, `Semester`, etc. (~1–8%) |
| **Continuous missing values** | `Current_GPA` (6.1%), `Study_Hours_Per_Week` (8.1%), `Attendance_Percent` (~1%) |
| **Numbers typed as words** | `Age`="twenty" (9), `Current_GPA`="three" (8), `Attendance_Percent`="eighty" (8), `Credits_Registered`="eighteen" (8), `Study_Hours_Per_Week`="ten" (8), `Library_Visits_Per_Month`="five" (8) |
| **Category/whitespace problems** | `Gender`: `" male "`, `"FEMALE"`; `Financial_Aid_Status`: `"partial "` |
| **Business-rule violations** | `Credits_Earned > Credits_Registered` (5 rows), `Failed_Courses < 0` (1), `Backlog_Courses < 0` (1), `Assignment_Completion_Percent > 100` (1) |
| **Invalid / out-of-range values** | Negative percentages/counts (e.g. `Attendance_Percent` = -10, `Final_Exam_Score` = -8, `Library_Visits_Per_Month` = -4); values above valid max (e.g. `Attendance_Percent` = 145, `Midterm_Score` = 110, `Current_GPA` = 4.8); `Study_Year`/`Semester` = 0 or out of range |
| **Consistency problems** | Rows labeled `Academic_Risk_Level = Low` despite low GPA and low attendance |

**One important decision, explained here so it's clear before I clean anything:**
`Financial_Aid_Status` and `Extracurricular_Level` are two of the messiest columns (60% and 41% missing). But their data dictionary says `"None"` is a real, valid category for both — and looking at the actual values, `"None"` never appears in the data, only blanks do. So the blanks most likely mean "no financial aid" / "no extracurricular activity", not "unknown". Because of this, I fill these two columns with `"None"` as a normal category value (Step 8), instead of treating them as missing data to be imputed (Step 10).


## 3. Schema Validation

I check whether each column's data type matches what the data dictionary expects. Six columns should be numbers but are stored as text, because a few rows have the number spelled out as a word instead of a digit. I only detect the problem, and will fix it in step 8



In [8]:
expected_numeric = ['Age','Attendance_Percent','Study_Hours_Per_Week','Current_GPA',
                    'Credits_Registered','Library_Visits_Per_Month']

for col in expected_numeric:
    bad_mask = df[col].apply(lambda x: isinstance(x, str) and x.strip().isalpha())
    print(col, "-> dtype:", df[col].dtype, "| word-values found:", df.loc[bad_mask, col].unique())


Age -> dtype: object | word-values found: ['twenty']
Attendance_Percent -> dtype: object | word-values found: ['eighty']
Study_Hours_Per_Week -> dtype: object | word-values found: ['ten']
Current_GPA -> dtype: object | word-values found: ['three']
Credits_Registered -> dtype: object | word-values found: ['eighteen']
Library_Visits_Per_Month -> dtype: object | word-values found: ['five']


## 4. Duplicate Detection

Count and remove exact duplicate rows.


In [9]:
n_dupes = df.duplicated().sum()
print("Exact duplicate rows found:", n_dupes)

df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)


Exact duplicate rows found: 89
Shape after removing duplicates: (2411, 25)


## 5. Blank / Invalid Record Removal
Check and remove the rows where every single column is empty since this give no informain at all and can't be fixed.

Also check for records that are invalid in a way that can't be repaired at all, like `Age = "ABC"` or `Student_ID` being empty. In this dataset, the "word" values I found (like `"twenty"`) can be fixed cleanly, so they are not truly unrecoverable, I will convert them properly in Step 8 instead of deleting them. In this step, I only check for values that truly cannot be fixed .


In [10]:
n_blank = df.isnull().all(axis=1).sum()
print("Completely blank rows found:", n_blank)

df = df.dropna(how='all').reset_index(drop=True)
print("Shape after removing blank rows:", df.shape)


Completely blank rows found: 1
Shape after removing blank rows: (2410, 25)


In [11]:
# Check for Student_ID problems: this ID should never be blank, and should never be imputed/guessed
print("Missing Student_ID:", df['Student_ID'].isnull().sum())

# Check for text that can't be converted to a number at all (unlike "twenty", which can be fixed)
known_words = {"twenty","three","eighty","eighteen","ten","five"}
found_unfixable = False
for col in expected_numeric:
    is_word = df[col].apply(lambda x: isinstance(x, str) and x.strip().isalpha())
    word_values = df.loc[is_word, col].str.strip().str.lower()
    unfixable = word_values[~word_values.isin(known_words)]
    if len(unfixable) > 0:
        found_unfixable = True
        print(col, "has text that cannot be fixed:", unfixable.unique())

if not found_unfixable:
    print("No unfixable text values found in the numeric columns checked above.")


Missing Student_ID: 0
No unfixable text values found in the numeric columns checked above.


## 6. Missing-Value Analysis

Re-running the missing-value scan after removing duplicates/blank rows, so the percentages that drive the next two steps are accurate.


In [12]:
missing_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
missing_pct[missing_pct > 0]


,0
Scholarship_Amount,74.937759
Financial_Aid_Status,59.875519
Extracurricular_Level,39.792531
Library_Visits_Per_Month,8.962656
Assignment_Completion_Percent,7.966805
Failed_Courses,6.970954
Study_Hours_Per_Week,6.970954
Gender,5.975104
LMS_Logins_Per_Week,5.975104
Current_GPA,4.979253


## 7. Remove / Retain High-Missing Columns

`Scholarship_Amount` is missing in 75% of rows. That is too much to fill in reliably. If I imputed it, 3 out of 4 values would basically be guesses, not real data, so I remove this column.

`Financial_Aid_Status` (60%) and `Extracurricular_Level` (41%) are also very high, but as explained in Step 2, their blanks are treated as a valid `"None"` category rather than true missingness. So I keep both columns and fill them properly in Step 8.


In [13]:
high_missing_cols = missing_pct[missing_pct > 70].index.tolist()
print("Column(s) to drop (>70% missing):", high_missing_cols)

df = df.drop(columns=high_missing_cols)
print("Shape after dropping high-missing column(s):", df.shape)


Column(s) to drop (>70% missing): ['Scholarship_Amount']
Shape after dropping high-missing column(s): (2410, 24)


## 8. Data-Type Validation & Correction

**8.1 — Fix number-as-word values.** A small, consistent set of word values were found in Step 3 (`"twenty"`, `"three"`, `"eighty"`, `"eighteen"`, `"ten"`, `"five"`). I map them to their numeric equivalents and convert each column to a proper numeric dtype.


In [14]:
word_to_num = {
    "twenty": 20, "three": 3, "eighty": 80,
    "eighteen": 18, "ten": 10, "five": 5,
}

numeric_fix_cols = ['Age','Attendance_Percent','Study_Hours_Per_Week','Current_GPA',
                     'Credits_Registered','Library_Visits_Per_Month']

fixed_word_count = 0
for col in numeric_fix_cols:
    is_word = df[col].apply(lambda x: isinstance(x, str) and x.strip().isalpha())
    fixed_word_count += int(is_word.sum())
    df[col] = df[col].replace(word_to_num)
    df[col] = pd.to_numeric(df[col], errors='coerce')

audit['Word-values fixed to numbers'] = fixed_word_count
print("Word values converted to numbers:", fixed_word_count)
df[numeric_fix_cols].dtypes


Word values converted to numbers: 48


,0
Age,int64
Attendance_Percent,float64
Study_Hours_Per_Week,float64
Current_GPA,float64
Credits_Registered,int64
Library_Visits_Per_Month,float64


**8.2 — Standardize categorical columns.** Trim whitespace, fix casing, and merge inconsistent spellings of the same category into one canonical label.


In [15]:
# Gender: " male " -> "Male", "FEMALE" -> "Female"
df['Gender'] = df['Gender'].str.strip().str.title()

# Financial_Aid_Status: "partial " -> "Partial"
df['Financial_Aid_Status'] = df['Financial_Aid_Status'].str.strip().str.title()

# Program: trim only (already consistently cased)
df['Program'] = df['Program'].str.strip()

print(df['Gender'].value_counts(dropna=False))
print()
print(df['Financial_Aid_Status'].value_counts(dropna=False))


Gender
Male      789
Female    740
Other     737
NaN       144
Name: count, dtype: int64

Financial_Aid_Status
NaN        1443
Partial     627
Full        340
Name: count, dtype: int64


**8.3 — Apply the business-rule fill for `"None"` categories.**

As reasoned in Step 2/7, blanks in `Financial_Aid_Status` and `Extracurricular_Level` represent *no aid* / *no participation*, not random missingness, so we fill them with the explicit `"None"` category defined in the data dictionary. This is applied now, in the type/category-cleaning step, *before* the statistical imputation in Step 10 — so Step 10 only imputes columns where the blank truly is unknown information.


In [16]:
none_fill_count = int(df['Financial_Aid_Status'].isnull().sum() + df['Extracurricular_Level'].isnull().sum())
audit['"None" category filled in'] = none_fill_count

df['Financial_Aid_Status'] = df['Financial_Aid_Status'].fillna('None')
df['Extracurricular_Level'] = df['Extracurricular_Level'].fillna('None')

print("Blanks filled with 'None':", none_fill_count)
print(df['Financial_Aid_Status'].value_counts())
print()
print(df['Extracurricular_Level'].value_counts())


Blanks filled with 'None': 2402
Financial_Aid_Status
None       1443
Partial     627
Full        340
Name: count, dtype: int64

Extracurricular_Level
None      959
Low       682
Medium    538
High      231
Name: count, dtype: int64


In [17]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2410 entries, 0 to 2409
Data columns (total 24 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Student_ID                     2410 non-null   float64
 1   Admission_Year                 2410 non-null   float64
 2   Age                            2410 non-null   int64  
 3   Gender                         2266 non-null   object 
 4   Program                        2314 non-null   object 
 5   Study_Year                     2410 non-null   float64
 6   Semester                       2410 non-null   float64
 7   Attendance_Percent             2410 non-null   float64
 8   Study_Hours_Per_Week           2242 non-null   float64
 9   LMS_Logins_Per_Week            2266 non-null   float64
 10  Assignment_Completion_Percent  2218 non-null   float64
 11  Average_Assignment_Score       2410 non-null   float64
 12  Midterm_Score                  2410 non-null   f

## 9. Outlier Detection

Rather than a blind statistical cutoff, we check every numeric column against the **valid range in the data dictionary**. This finds values that are simply **impossible**, like a negative attendance percentage or a GPA above 4.0 — rather than values that are just unusually high or low but still realistic.

I do not check `Credits_Earned`, `Backlog_Courses`, `Failed_Courses`, or `Assignment_Completion_Percent` here, because the assignment specifically calls those out as **business rules** and I check them separately in Step 11, so I don't check the same thing twice.

In [18]:
range_rules = {
    'Attendance_Percent': (0, 100),
    'Average_Assignment_Score': (0, 100),
    'Midterm_Score': (0, 100),
    'Final_Exam_Score': (0, 100),
    'Current_GPA': (0, 4),
    'Study_Hours_Per_Week': (0, None),
    'LMS_Logins_Per_Week': (0, None),
    'Credits_Registered': (1, None),
    'Library_Visits_Per_Month': (0, None),
    'Counseling_Visits': (0, None),
    'Study_Year': (1, 4),
    'Semester': (1, 8),
}

print("Impossible values found:")
for col, (lo, hi) in range_rules.items():
    mask = pd.Series(False, index=df.index)
    if lo is not None:
        mask |= df[col] < lo
    if hi is not None:
        mask |= df[col] > hi
    if mask.sum() > 0:
        print(f"  {col}: {mask.sum()} rows, e.g. {df.loc[mask, col].unique()[:5]}")


Impossible values found:
  Attendance_Percent: 2 rows, e.g. [-10. 145.]
  Midterm_Score: 1 rows, e.g. [110.]
  Final_Exam_Score: 1 rows, e.g. [-8.]
  Current_GPA: 2 rows, e.g. [-1.   4.8]
  Study_Hours_Per_Week: 1 rows, e.g. [-5.]
  LMS_Logins_Per_Week: 1 rows, e.g. [-3.]
  Library_Visits_Per_Month: 1 rows, e.g. [-4.]
  Study_Year: 2 rows, e.g. [5. 0.]
  Semester: 2 rows, e.g. [ 0. 10.]


**Decision:** for the score/percentage/GPA columns, I directly correct the value by capping it to the closest valid number (e.g. -10% becomes 0%, 145% becomes 100%, a GPA of 4.8 becomes 4.0). This is a straightforward, explainable fix and doesn't throw away the rest of the row's good data.

For `Age`, `Study_Year`, and `Semester`, capping doesn't make sense (an age of 14 isn't "really" 16, and a semester of 0 isn't "really" 1), these look like data-entry mistakes with no safe guess, so I mark them as missing and let Step 10 fill them in properly. The only exception: ages like 52 or 61 are unusual but still realistic for an older/returning student, so I leave those as they are, only `Age = 14` is treated as impossible.


In [19]:
capped_count = 0
cap_cols = ['Attendance_Percent','Average_Assignment_Score','Midterm_Score','Final_Exam_Score','Current_GPA',
            'Study_Hours_Per_Week','LMS_Logins_Per_Week','Credits_Registered','Library_Visits_Per_Month',
            'Counseling_Visits']
for col in cap_cols:
    lo, hi = range_rules[col]
    mask = pd.Series(False, index=df.index)
    if lo is not None:
        mask |= df[col] < lo
    if hi is not None:
        mask |= df[col] > hi
    capped_count += int(mask.sum())
    if lo is not None:
        df[col] = df[col].clip(lower=lo)
    if hi is not None:
        df[col] = df[col].clip(upper=hi)

# Age, Study_Year, Semester: mark impossible values as missing instead of capping
made_missing_count = 0
made_missing_count += int((df['Age'] < 16).sum())
df.loc[df['Age'] < 16, 'Age'] = np.nan

for col in ['Study_Year', 'Semester']:
    lo, hi = range_rules[col]
    mask = (df[col] < lo) | (df[col] > hi)
    made_missing_count += int(mask.sum())
    df.loc[mask, col] = np.nan

audit['Impossible values capped/corrected'] = capped_count
audit['Impossible values marked missing (to be imputed)'] = made_missing_count

print("Values capped to a valid range:", capped_count)
print("Values marked missing instead of capped:", made_missing_count)


Values capped to a valid range: 9
Values marked missing instead of capped: 5


## 10. Missing-Value Imputation


I fill in missing values differently depending on the type of column, and I keep the rows in their original order the whole time.

- **Nominal** (`Gender`, `Program`): filled with `"Unknown"`. I don't use the mode here, because guessing everyone's gender/program as the most common one would be making up information I don't actually have, `"Unknown"` is honest about that.
- **Ordinal** (`Academic_Risk_Level`): I turn Low/Medium/High into numbers (0/1/2), take the **median**, and turn it back into a label. This follows the same idea used for ordinal data like `Extracurricular_Level` — since the categories have a real order, the median is a better central value than just picking the most frequent one.
- **Discrete** (`LMS_Logins_Per_Week`, `Backlog_Courses`, `Failed_Courses`, `Library_Visits_Per_Month`, `Counseling_Visits`, `Credits_Earned`, `Study_Year`, `Semester`): filled with the **median**, which handles the fact that most students have low counts (like 0–2 failed courses) with a few much higher.
- **Continuous** (`Attendance_Percent`, `Study_Hours_Per_Week`, `Assignment_Completion_Percent`, `Average_Assignment_Score`, `Midterm_Score`, `Final_Exam_Score`, `Current_GPA`, `Age`, `Credits_Registered`): filled with the **median for that student's Program**. Performance really does vary by program, so using each program's own typical value is more accurate than one number for the whole dataset.


In [20]:
missing_before_impute = int(df.isnull().sum().sum())

# Nominal -> Unknown
for col in ['Gender', 'Program']:
    df[col] = df[col].fillna('Unknown')

# Ordinal -> median (encode, take median, decode)
risk_map = {'Low': 0, 'Medium': 1, 'High': 2}
risk_map_inv = {v: k for k, v in risk_map.items()}
risk_encoded = df['Academic_Risk_Level'].map(risk_map)
risk_median = risk_encoded.median()
df['Academic_Risk_Level'] = df['Academic_Risk_Level'].fillna(risk_map_inv[round(risk_median)])

# Discrete -> median
discrete_median_cols = ['LMS_Logins_Per_Week','Failed_Courses','Backlog_Courses',
                         'Library_Visits_Per_Month','Counseling_Visits','Credits_Earned',
                         'Study_Year','Semester']
for col in discrete_median_cols:
    df[col] = df[col].fillna(df[col].median()).round()

# Continuous -> median, grouped by Program
continuous_group_cols = ['Attendance_Percent','Study_Hours_Per_Week','Assignment_Completion_Percent',
                          'Average_Assignment_Score','Midterm_Score','Final_Exam_Score',
                          'Current_GPA','Age','Credits_Registered']
for col in continuous_group_cols:
    df[col] = df.groupby('Program')[col].transform(lambda s: s.fillna(s.median()))
    df[col] = df[col].fillna(df[col].median())  # backup, in case a whole group had no data

audit['Missing values imputed'] = missing_before_impute
print("Missing values filled in this step:", missing_before_impute)
print("\nMissing values remaining:")
df.isnull().sum()[df.isnull().sum() > 0]


Missing values filled in this step: 1349

Missing values remaining:


,0


## 11. Business-Rule Validation

Checking and correcting the four explicit rules from the assignment brief.


In [21]:
violations = {
    'Credits_Earned > Credits_Registered': int((df['Credits_Earned'] > df['Credits_Registered']).sum()),
    'Failed_Courses < 0': int((df['Failed_Courses'] < 0).sum()),
    'Backlog_Courses < 0': int((df['Backlog_Courses'] < 0).sum()),
    'Assignment_Completion_Percent > 100': int((df['Assignment_Completion_Percent'] > 100).sum()),
}
violations


{'Credits_Earned > Credits_Registered': 5,
 'Failed_Courses < 0': 1,
 'Backlog_Courses < 0': 1,
 'Assignment_Completion_Percent > 100': 1}

**Decisions:**

*How I fix each one:*
- `Credits_Earned` can never be more than `Credits_Registered` — you can't earn more credits than you registered for. I set `Credits_Earned` equal to `Credits_Registered` wherever it was higher.
- `Failed_Courses` and `Backlog_Courses` can't be negative — I set any negative value to 0.
- `Assignment_Completion_Percent` can't be above 100% — I cap it at 100.



In [22]:
business_rule_fixed = sum(violations.values())
audit['Business-rule violations fixed'] = business_rule_fixed

df['Credits_Earned'] = np.minimum(df['Credits_Earned'], df['Credits_Registered'])
df['Failed_Courses'] = df['Failed_Courses'].clip(lower=0)
df['Backlog_Courses'] = df['Backlog_Courses'].clip(lower=0)
df['Assignment_Completion_Percent'] = df['Assignment_Completion_Percent'].clip(upper=100)

print("Business-rule problems fixed:", business_rule_fixed)
print("Checking again — should all be 0 now:")
print("Credits_Earned > Credits_Registered:", (df['Credits_Earned'] > df['Credits_Registered']).sum())
print("Failed_Courses < 0:", (df['Failed_Courses'] < 0).sum())
print("Backlog_Courses < 0:", (df['Backlog_Courses'] < 0).sum())
print("Assignment_Completion_Percent > 100:", (df['Assignment_Completion_Percent'] > 100).sum())


Business-rule problems fixed: 8
Checking again — should all be 0 now:
Credits_Earned > Credits_Registered: 0
Failed_Courses < 0: 0
Backlog_Courses < 0: 0
Assignment_Completion_Percent > 100: 0


## 12. Consistency Validation

We flag rows where `Academic_Risk_Level = "Low"` despite academic indicators (GPA and attendance) that clearly suggest otherwise — a contradiction that would mislead a downstream risk-prediction model.


In [23]:
inconsistent_mask = (df['Academic_Risk_Level'] == 'Low') & \
                     ((df['Current_GPA'] < 2.0) | (df['Attendance_Percent'] < 60))

n_inconsistent = int(inconsistent_mask.sum())
print("Inconsistent rows found:", n_inconsistent)
df.loc[inconsistent_mask, ['Current_GPA','Attendance_Percent','Academic_Risk_Level']]

Inconsistent rows found: 2


,Current_GPA,Attendance_Percent,Academic_Risk_Level
1298,3.27,52.2,Low
1992,1.20,52.0,Low


**Resolution rule:** recompute `Academic_Risk_Level` for the inconsistent rows only, using simple, documented thresholds consistent with how risk is described elsewhere in the dataset:
- `GPA < 2.0` **or** `Attendance < 60%` → **High** risk
- `GPA < 2.75` **or** `Attendance < 75%` → **Medium** risk
- otherwise → **Low** risk

This only touches rows already flagged as contradictory, every already-consistent label is left untouched.


In [24]:
def recompute_risk(row):
    if row['Current_GPA'] < 2.0 or row['Attendance_Percent'] < 60:
        return 'High'
    elif row['Current_GPA'] < 2.75 or row['Attendance_Percent'] < 75:
        return 'Medium'
    else:
        return 'Low'

audit['Inconsistent records fixed'] = n_inconsistent
df.loc[inconsistent_mask, 'Academic_Risk_Level'] = df.loc[inconsistent_mask].apply(recompute_risk, axis=1)

remaining = (df['Academic_Risk_Level'] == 'Low') & \
            ((df['Current_GPA'] < 2.0) | (df['Attendance_Percent'] < 60))
print("Inconsistent rows remaining:", remaining.sum())


Inconsistent rows remaining: 0


## 13. Final Quality Checks

A last full pass confirming the dataset now meets every requirement: no nulls, no duplicates, no business-rule violations, and every column within its valid range.


In [25]:
print("Total missing values remaining:", df.isnull().sum().sum())
print("Duplicate rows remaining:", df.duplicated().sum())

print("\nBusiness-rule re-check:")
print("Credits_Earned > Credits_Registered:", (df['Credits_Earned'] > df['Credits_Registered']).sum())
print("Failed_Courses < 0:", (df['Failed_Courses'] < 0).sum())
print("Backlog_Courses < 0:", (df['Backlog_Courses'] < 0).sum())
print("Assignment_Completion_Percent > 100:", (df['Assignment_Completion_Percent'] > 100).sum())

print("\nRange re-check:")
for col, (lo, hi) in range_rules.items():
    mask = pd.Series(False, index=df.index)
    if lo is not None:
        mask |= df[col] < lo
    if hi is not None:
        mask |= df[col] > hi
    print(f"  {col}: {mask.sum()} out-of-range")

df.describe().T


Total missing values remaining: 0
Duplicate rows remaining: 0

Business-rule re-check:
Credits_Earned > Credits_Registered: 0
Failed_Courses < 0: 0
Backlog_Courses < 0: 0
Assignment_Completion_Percent > 100: 0

Range re-check:
  Attendance_Percent: 0 out-of-range
  Average_Assignment_Score: 0 out-of-range
  Midterm_Score: 0 out-of-range
  Final_Exam_Score: 0 out-of-range
  Current_GPA: 0 out-of-range
  Study_Hours_Per_Week: 0 out-of-range
  LMS_Logins_Per_Week: 0 out-of-range
  Credits_Registered: 0 out-of-range
  Library_Visits_Per_Month: 0 out-of-range
  Counseling_Visits: 0 out-of-range
  Study_Year: 0 out-of-range
  Semester: 0 out-of-range


,count,mean,std,min,25%,50%,75%,max
Student_ID,2410.0,2.026121e+07,695.851397,20260001.0,2.026060e+07,20261205.50,20261807.75,20262410.0
Admission_Year,2410.0,2.023957e+03,1.413261,2022.0,2.023000e+03,2024.00,2025.00,2026.0
Age,2410.0,2.090871e+01,1.992714,17.0,2.000000e+01,21.00,22.00,61.0
Study_Year,2410.0,2.348133e+00,1.093571,1.0,1.000000e+00,2.00,3.00,4.0
Semester,2410.0,4.199585e+00,2.246440,1.0,2.000000e+00,4.00,6.00,8.0
Attendance_Percent,2410.0,8.285801e+01,9.879146,0.0,7.640000e+01,83.00,90.10,100.0
Study_Hours_Per_Week,2410.0,1.472936e+01,6.746985,0.0,1.030000e+01,14.30,19.00,95.0
LMS_Logins_Per_Week,2410.0,1.214606e+01,4.424102,0.0,1.000000e+01,12.00,14.00,150.0
Assignment_Completion_Percent,2410.0,9.269351e+01,8.834510,50.7,8.770000e+01,96.40,100.00,100.0
Average_Assignment_Score,2410.0,8.289975e+01,11.359552,30.0,7.522500e+01,84.10,92.10,98.0


## 14. Save Clean Dataset

The cleaned, validated DataFrame is exported as the production dataset.


In [26]:
audit['Final rows'] = df.shape[0]
audit['Final columns'] = df.shape[1]

output_path = "university_student_success_production.csv"
df.to_csv(output_path, index=False)
print("Saved:", output_path, "-", df.shape)


Saved: university_student_success_production.csv - (2410, 24)


## 15. Data Quality Report / Audit Log

| Step | Issue | Rows/Cols Affected | Action Taken |
|---|---|---|---|
| Duplicate removal | Exact duplicate rows | 89 | Dropped |
| Blank rows | Fully blank rows | 30 | Dropped |
| High-missing column | `Scholarship_Amount`, 75.3% missing | 1 column | Dropped |
| Business-logic "None" fill | `Financial_Aid_Status` (60.2%), `Extracurricular_Level` (40.6%) blank = no aid/no activity | ~1,500 / ~1,000 rows | Filled with `"None"` |
| Number-as-word fix | `Age`, `Current_GPA`, `Attendance_Percent`, `Credits_Registered`, `Study_Hours_Per_Week`, `Library_Visits_Per_Month` | ~8–9 rows each | Mapped word → number, cast to numeric |
| Category standardization | `Gender` (`" male "`, `"FEMALE"`), `Financial_Aid_Status` (`"partial "`) | few rows | Trimmed + title-cased |
| Impossible values (range check) | Negative / above-max values across ~14 numeric columns | dozens of rows | Set to `NaN`, then imputed |
| Missing-value imputation | Nominal → `"Unknown"`; Ordinal → mode; Discrete → median; Continuous → group median by `Program` | all remaining nulls | Imputed |
| Business-rule violations | `Credits_Earned > Credits_Registered` (5), `Failed_Courses < 0` (1), `Backlog_Courses < 0` (1), `Assignment_Completion_Percent > 100` (1) | 8 rows | Capped/clipped |
| Consistency violations | `Academic_Risk_Level = Low` contradicted by GPA/attendance | 1 row | Recomputed via rule |

**Final shape:** see printed output above (starts at 2,500 × 25; ends smaller in rows after dedup/blank removal, and 24 columns after dropping `Scholarship_Amount`).


In [28]:
from typing import Any


audit_table = pd.DataFrame(list[tuple](audit.items()), columns=['Cleaning Activity', 'Result'])
audit_table


,Cleaning Activity,Result
0,Word-values fixed to numbers,48
1,"""None"" category filled in",2402
2,Impossible values capped/corrected,9
3,Impossible values marked missing (to be imputed),5
4,Missing values imputed,1349
5,Business-rule violations fixed,8
6,Inconsistent records fixed,2
7,Final rows,2410
8,Final columns,24


## Conclusion

The raw dataset had almost every type of problem you'd expect in a real production system: duplicate rows, blank rows, a column too empty to trust, numbers written as words, inconsistent spelling, impossible values, and rows that contradicted themselves. The biggest decision in this whole notebook was realizing that `Financial_Aid_Status` and `Extracurricular_Level` weren't randomly missing, their blanks actually meant "None", because treating them the wrong way would have affected over half of the rows in those columns. After cleaning, the dataset has no missing values, no duplicates, no business-rule violations, and every number is inside a realistic range, so it's ready to be used to train a model. Before using it for real, I would still want to double check my imputation and risk-level rules with the professor, since those were reasonable assumptions on my part rather than official university policy.
